### 구조화된 출력 파서
- LLM에 대한 답변을 딕셔너리 형식으로 정의하고, 키와 값의 쌍으로 여러 필드를 반환하는 데이터 구조화에 유용
- 로컬 모델에서는 원하는 형식으로 데이터가 잘 나오지 않는 경우가 많기 때문에 이 파서를 이용하면 좀 더 안정적인 출력 형식을 얻을 수 있습니다.

In [1]:
import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field

# 모델
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 프롬프트
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
    FewShotPromptTemplate,
    FewShotChatMessagePromptTemplate,
)

# 예시 선택기 / 벡터스토어
from langchain_core.example_selectors import (
    MaxMarginalRelevanceExampleSelector,
    SemanticSimilarityExampleSelector,
)
from langchain_chroma import Chroma

# 출력 파서
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser

# teddynote 유틸
from langchain_teddynote import logging
from langchain_teddynote.messages import stream_response


# ── 환경 설정 ──────────────────────────────
load_dotenv()
logging.langsmith("test0914")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("OpenAI 키 로드됨 : ", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith 키 로드됨 : ", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith 프로젝트 : ", os.getenv("LANGSMITH_PROJECT"))

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914
OpenAI 키 로드됨 :  True
LangSmith 키 로드됨 :  True
LangSmith 프로젝트 :  test0914


In [3]:
response_schemas = [
    ResponseSchema(name="answer", description="사용자의 질문에 대한 답변"),
    ResponseSchema(name="source", description="사용자의 질문에 답하기 위해 사용된 '출처', '웹사이트 주소' 이어야 합니다.")
]

# 로컬모델에서는 인텔리전스가 부족할 수 있기 때문에 description 의 내용을 영어로 작성하는게 좋음

In [ ]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# answer 와 source 라는 두 항목을 포함한 형식화된 출력값이 생성됨

In [5]:
print(output_parser.get_format_instructions())

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"answer": string  // 사용자의 질문에 대한 답변
	"source": string  // 사용자의 질문에 답하기 위해 사용된 '출처', '웹사이트 주소' 이어야 합니다.
}
```


In [7]:
format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template="answer the users question as best as possible.\n{format_instructions}\n{question}",

    input_variables=["question"],
    partial_variables={"format_instructions": format_instructions}
)

In [8]:
model = ChatOpenAI(temperature=0)

chain = prompt | model | output_parser

In [9]:
chain.invoke({"question": "나이지리아의 수도는 어디인가요 ?"})

{'answer': '나이지리아의 수도는 아부자입니다.',
 'source': 'https://ko.wikipedia.org/wiki/%EB%82%98%EC%9D%B4%EC%A7%80%EB%A6%AC%EC%95%84'}